# Build the homepage, CVs and publication list

This notebook runs the same steps as `./update.sh`, one cell at a time, so you can look at the
record before building. Everything is explained in [docs/BUILD.md](docs/BUILD.md).

**The record:** `data/records.csv` (one row per item). **The inbox:** `data/new_entries.yaml`.
**Settings and texts:** `data/settings.yaml`.

In [ ]:
import sys, csv, collections
sys.path.insert(0, "tools")
import build as B

rows = B.read_records()
print(len(rows), "rows in data/records.csv")
for sec, n in collections.Counter(r["section"] for r in rows).items():
    print(f"  {sec:<11} {n}")

## 1. Add new items (optional)

Either edit `data/new_entries.yaml` directly (copy blocks from `data/template.yaml`), or use
the helper below. A DOI is usually enough for a paper; blank fields are fine.

In [ ]:
import yaml

def add_to_inbox(**entry):
    """add_to_inbox(section="article", doi="10.1029/...", cv_short="yes")"""
    doc = yaml.safe_load(B.INBOX.read_text(encoding="utf-8")) or {}
    doc["entries"] = (doc.get("entries") or []) + [entry]
    text = B.INBOX_HEADER.replace("entries: []\n", "")
    B.INBOX.write_text(text + yaml.safe_dump(doc, allow_unicode=True, sort_keys=False), encoding="utf-8")
    print("Inbox now has", len(doc["entries"]), "entr(y/ies)")

# Example (remove the # to use):
# add_to_inbox(section="article", doi="10.1029/2024GL110098", cv_short="yes")

## 2. Check: import the inbox, fill from DOIs, validate

Nothing is written to the homepage or PDFs yet. Errors are listed with the CSV line number.

In [ ]:
B.run(check=True)

## 3. See what will appear where

In [ ]:
def show(output, section):
    for r in B.select(B.read_records(), output, section):
        print(f"{B.years(r):>14}  {r['id']:<26} {B.val(r, 'title')[:80]}")

show("cv_short", "article")   # outputs: homepage, cv_long, cv_short, reflist

## 4. Build

Writes `index.html`, `publications/references.bib` and the `.tex` files, then compiles the PDFs
(needs XeLaTeX). Use `only=[...]` to build a subset, `pdf=False` to skip LaTeX.

In [ ]:
B.run()                                   # everything switched on in settings.yaml
# B.run(only=["homepage", "cv_short"])    # just these
# B.run(pdf=False)                        # no LaTeX

## 5. Preview the one-page CV (optional, needs `pip install pymupdf`)

In [ ]:
try:
    import pymupdf
    from IPython.display import Image, display
    page = pymupdf.open("cv/Tobias_Staal_CV_short.pdf")[0]
    display(Image(page.get_pixmap(dpi=70).tobytes("png")))
except ImportError:
    print("pip install pymupdf to preview here; or open cv/Tobias_Staal_CV_short.pdf")

## 6. Publish

Commit and push (or run `./update.sh "message"` in a terminal, which builds and pushes in one go).
If you push only changes in `data/`, GitHub Actions rebuilds everything for you.

In [ ]:
# !git add -A && git commit -m "Update CV and publications" && git push origin main